# Unsupervised Learning with Gaussian Processes

$$
$$

<!-- CIP-0012 Track A: no-op outside HTML (overridden in talk-macros-html.gpp). -->
<!-- Class-test timer: no-op outside slides HTML (overridden in talk-macros-slides-html.gpp). -->
<!-- CIP-0016: mdpp expands \diagramcredit before gpp; leftover calls are empty. -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- CIP-0012 Track A: content script load. Once-guard with alphanumeric
     names stays in the surrounding ifndef wrapper (GPP ifndef limitation). -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## Gaussian Process Summer School notebook setup

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gpss/includes/gpss-notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gpss/includes/gpss-notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We install packages used across the notebooks for data loading, plotting
helpers, and Gaussian process modelling.

In [ ]:
import importlib.util

In [ ]:
def install_command(imp):
    import shutil
    import sys
    packages = {
        'daft': 'daft-pgm',
        'mlai': 'git+https://github.com/lawrennd/mlai.git',
        'pods': 'pods',
        'PyDeepGP': 'git+https://github.com/SheffieldML/PyDeepGP.git',
        'notutils': 'git+https://github.com/lawrennd/notutils.git',
        'pods' : 'git+https://github.com/lawrennd/ods.git',
        'lamd': 'lamd',
        'GPy': 'gpy>=1.15',
    'emukit': 'emukit',
    }

    def _installer():
        if shutil.which('uv'):
            return 'uv pip install --python "' + sys.executable + '"'
        return '"' + sys.executable + '" -m pip install'

    def _ver_tuple(v):
        parts = []
        for p in v.split('.')[:3]:
            try:
                parts.append(int(''.join(ch for ch in p if ch.isdigit()) or '0'))
            except ValueError:
                parts.append(0)
        while len(parts) < 3:
            parts.append(0)
        return tuple(parts)

    def _below_floor(spec):
        if '>=' not in spec:
            return False
        name, floor = spec.split('>=', 1)
        name = name.strip()
        floor = floor.strip().split(',')[0].strip()
        try:
            import importlib.metadata as md
            cur = md.version(name)
        except Exception:
            return True
        return _ver_tuple(cur) < _ver_tuple(floor)

    spec = packages[imp] if imp in packages else imp
    missing = importlib.util.find_spec(imp) is None
    if missing or _below_floor(spec):
        return _installer() + ' "' + spec + '"'
    return 'echo "Package ' + imp + ' is already installed."'

In [ ]:
cmd = install_command('pods')

In [ ]:
%system {cmd}

In [ ]:
cmd = install_command('notutils')

In [ ]:
%system {cmd}

In [ ]:
cmd = install_command('mlai')

In [ ]:
%system {cmd}

In [ ]:
import matplotlib.pyplot as plt
import shutil

if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False
else:
    plt.rcParams['text.usetex'] = True
    plt.rcParams['text.latex.preamble'] = (
        r'\usepackage{amsmath}\usepackage{amssymb}\usepackage{mathrsfs}'
    )

plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

In [ ]:
cmd = install_command('GPy')

In [ ]:
%system {cmd}

## GPy: A Gaussian Process Framework in Python

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/gpy-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/gpy-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Gaussian processes are a flexible tool for non-parametric analysis with
uncertainty. The GPy software was started in Sheffield to provide a easy
to use interface to GPs. One which allowed the user to focus on the
modelling rather than the mathematics.

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gp/gpy.png" style="width:70%; height:auto; display:block;">

Figure: <i>GPy is a BSD licensed software code base for implementing
Gaussian process models in Python. It is designed for teaching and
modelling. We welcome contributions which can be made through the GitHub
repository <https://github.com/SheffieldML/GPy></i>

GPy is a BSD licensed software code base for implementing Gaussian
process models in python. This allows GPs to be combined with a wide
variety of software libraries.

The software itself is available on
[GitHub](https://github.com/SheffieldML/GPy) and the team welcomes
contributions.

The aim for GPy is to be a probabilistic-style programming language,
i.e., you specify the model rather than the algorithm. As well as a
large range of covariance functions the software allows for non-Gaussian
likelihoods, multivariate outputs, dimensionality reduction and
approximations for larger data sets.

The documentation for GPy can be found
[here](https://gpy.readthedocs.io/en/latest/).

## Probabilistic PCA

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/probabilistic-pca.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/probabilistic-pca.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This linear relationship between the observed data and the latent
variables is at the heart of Hotelling’s original formulation of PCA. It
explicitly models the idea that the observed high-dimensional data is
generated from a lower-dimensional set of latent variables, with some
added noise. This perspective aligns PCA more closely with factor
analysis and highlights its nature as a latent variable model.

Probabilistic PCA (PPCA) provides a probabilistic interpretation of PCA
by explicitly modeling the generative process that creates the observed
data. The model assumes that each high-dimensional data point
$\mathbf{ y}_{i,:}$ is generated from a lower-dimensional latent
variable $\mathbf{ z}_{i,:}$ through a linear transformation with added
Gaussian noise: $$
\mathbf{ y}_{i,:} = \mathbf{W}\mathbf{ z}_{i,:} + \boldsymbol{ \epsilon}_{i,:}
$$ where
$\boldsymbol{ \epsilon}_{i,:} \sim \mathscr{N}\left(\mathbf{0},\sigma^2\mathbf{I}\right)$
represents isotropic Gaussian noise. The model places a standard
Gaussian prior over the latent variables: $$
p(\mathbf{ z}_{i,:}) = \mathscr{N}\left(\mathbf{ z}_{i,:}|\mathbf{0},\mathbf{I}\right)
$$ Given these assumptions, the conditional probability of observing a
data point given its latent representation is: $$
p(\mathbf{ y}_{i,:}|\mathbf{ z}_{i,:},\mathbf{W}) = \mathscr{N}\left(\mathbf{ y}_{i,:}|\mathbf{W}\mathbf{ z}_{i,:},\sigma^2\mathbf{I}\right)
$$ By integrating out the latent variables, we obtain the marginal
likelihood of the data: $$
p(\mathbf{ y}_{i,:}|\mathbf{W}) = \mathscr{N}\left(\mathbf{ y}_{i,:}|\mathbf{0},\mathbf{W}\mathbf{W}^{\top}+\sigma^{2}\mathbf{I}\right)
$$ This probabilistic formulation, developed by Tipping and Bishop
(Tipping and Bishop, 1999a), provides a principled framework that not
only recovers classical PCA as a special case when $\sigma^2 \to 0$, but
also enables extensions like handling missing data and mixture models.

In [ ]:
cmd = install_command('daft')

In [ ]:
%system {cmd}

In [ ]:
from mlai import plot
import mlai
from matplotlib import pyplot as plt

In [ ]:
pgm = plot.ppca_graphical_model()
filename = mlai.filename_join('ppca_graphical_model.svg', directory='./dimred')
pgm.render().figure.savefig(filename, transparent=True)

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/ppca_graphical_model.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Graphical model representing probabilistic PCA.</i>

## Probabilistic PCA

In 1997 [Tipping and
Bishop](http://research.microsoft.com/pubs/67218/bishop-ppca-jrss.pdf)
(Tipping and Bishop, 1999b) and
[Roweis](https://www.cs.nyu.edu/~roweis/papers/empca.pdf) (Roweis, n.d.)
independently revisited Hotelling’s model and considered the case where
the noise variance was finite, but *shared* across all output dimensons.
Their model can be thought of as a factor analysis where $$
\boldsymbol{\Sigma} = \sigma^2 \mathbf{I}.
$$ This leads to a marginal likelihood of the form $$
p(\mathbf{Y}|\mathbf{W}, \sigma^2)
= \prod_{i=1}^n\mathscr{N}\left(\mathbf{ y}_{i, :}|\mathbf{0},\mathbf{W}\mathbf{W}^\top + \sigma^2 \mathbf{I}\right)
$$ where the limit of $\sigma^2\rightarrow 0$ is *not* taken. This
defines a proper probabilistic model. Tippping and Bishop then went on
to prove that the *maximum likelihood* solution of this model with
respect to $\mathbf{W}$ is given by an eigenvalue problem. In the
probabilistic PCA case the eigenvalues and eigenvectors are given as
follows. $$
\mathbf{W}= \mathbf{U}\mathbf{L} \mathbf{R}^\top
$$ where $\mathbf{U}$ is the eigenvectors of the empirical covariance
matrix $$
\mathbf{S}= \sum_{i=1}^n(\mathbf{ y}_{i, :} - \boldsymbol{ \mu})(\mathbf{ y}_{i,:} - \boldsymbol{ \mu})^\top,
$$ which can be written
$\mathbf{S}= \frac{1}{n} \mathbf{Y}^\top\mathbf{Y}$ if the data is zero
mean. The matrix $\mathbf{L}$ is diagonal and is dependent on the
*eigenvalues* of $\mathbf{S}$, $\boldsymbol{\Lambda}$. If the $i$th
diagonal element of this matrix is given by $\lambda_i$ then the
corresponding element of $\mathbf{L}$ is $$
\ell_i = \sqrt{\lambda_i - \sigma^2}
$$ where $\sigma^2$ is the noise variance. Note that if $\sigma^2$ is
larger than any particular eigenvalue, then that eigenvalue (along with
its corresponding eigenvector) is *discarded* from the solution.

## PPCA as Manifold Learning

Probabilistic PCA can be viewed as a simple manifold learning algorithm.
It assumes that:

1.  The data lies near a linear manifold (subspace) of the
    high-dimensional space
2.  The deviation from this manifold is Gaussian noise
3.  The intrinsic dimensionality is specified by the number of retained
    components

This view helps explain why PPCA works well when the manifold hypothesis
holds and the manifold is approximately linear. When the manifold is
nonlinear, we need more sophisticated methods like kernel PCA or neural
network-based approaches.

## Python Implementation of Probabilistic PCA

We will now implement this algorithm in python.

In [ ]:
import numpy as np

In [ ]:
from mlai import ppca_eig
import inspect
file_path = inspect.getfile(ppca_eig)

In [ ]:
%load -s ppca_eig {file_path}

In [ ]:
ppca = ppca_eig

In practice we may not wish to compute the eigenvectors of the
covariance matrix directly. This is because it requires us to estimate
the covariance, which involves a sum of squares term, before estimating
the eigenvectors. We can estimate the eigenvectors directly either
through [QR
decomposition](http://en.wikipedia.org/wiki/QR_decomposition) or
[singular value
decomposition](http://en.wikipedia.org/wiki/Singular_value_decomposition).
We saw a similar issue arise when , where we also wished to avoid
computation of $\mathbf{Z}^\top\mathbf{Z}$ (or in the case of
$\boldsymbol{\Phi}^\top\boldsymbol{\Phi}$).

## Posterior for Principal Component Analysis

Under the latent variable model justification for principal component
analysis, we are normally interested in inferring something about the
latent variables given the data. This is the distribution, $$
p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :})
$$ for any given data point. Determining this density turns out to be
very similar to the approach for determining the Bayesian posterior of
$\mathbf{ w}$ in Bayesian linear regression, only this time we place the
prior density over $\mathbf{ z}_{i, :}$ instead of $\mathbf{ w}$. The
posterior is proportional to the joint density as follows, $$
p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :}) \propto p(\mathbf{ y}_{i,
:}|\mathbf{W}, \mathbf{ z}_{i, :}, \sigma^2) p(\mathbf{ z}_{i, :})
$$ And as in the Bayesian linear regression case we first consider the
log posterior, $$
\log p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :}) = \log p(\mathbf{ y}_{i, :}|\mathbf{W},
\mathbf{ z}_{i, :}, \sigma^2) + \log p(\mathbf{ z}_{i, :}) + \text{const}
$$ where the constant is not dependent on $\mathbf{ z}$. As before we
collect the quadratic terms in $\mathbf{ z}_{i, :}$ and we assemble them
into a Gaussian density over $\mathbf{ z}$. $$
\log p(\mathbf{ z}_{i, :} | \mathbf{ y}_{i, :}) =
-\frac{1}{2\sigma^2} (\mathbf{ y}_{i, :} - \mathbf{W}\mathbf{ z}_{i,
:})^\top(\mathbf{ y}_{i, :} - \mathbf{W}\mathbf{ z}_{i, :}) - \frac{1}{2}
\mathbf{ z}_{i, :}^\top \mathbf{ z}_{i, :} + \text{const}
$$

### Exercise 1

Multiply out the terms in the brackets. Then collect the quadratic term
and the linear terms together. Show that the posterior has the form $$
\mathbf{ z}_{i, :} | \mathbf{W}\sim \mathscr{N}\left(\boldsymbol{ \mu}_x,\mathbf{C}_x\right)
$$ where $$
\mathbf{C}_x = \left(\sigma^{-2}
\mathbf{W}^\top\mathbf{W}+ \mathbf{I}\right)^{-1}
$$ and $$
\boldsymbol{ \mu}_x
= \mathbf{C}_x \sigma^{-2}\mathbf{W}^\top \mathbf{ y}_{i, :} 
$$ Compare this to the posterior for the Bayesian linear regression from
last week, do they have similar forms? What matches and what differs?

### Exercise 1 Answer

Write your answer to Exercise 1 here

## Python Implementation of the Posterior

Now let’s implement the system in code.

### Exercise 2

Use the values for $\mathbf{W}$ and $\sigma^2$ you have computed, along
with the data set $\mathbf{Y}$ to compute the posterior density over
$\mathbf{Z}$. Write a function of the form

``` python
mu_x, C_x = posterior(Y, W, sigma2)
```

where `mu_x` and `C_x` are the posterior mean and posterior covariance
for the given $\mathbf{Y}$.

Don’t forget to subtract the mean of the data `Y` inside your function
before computing the posterior: remember we assumed at the beginning of
our analysis that the data had been centred (i.e. the mean was removed).

In [ ]:
# Write your answer to Exercise 2 here


# Answer Code
# Write code for you answer to this exercise in this box
# Do not delete these comments, otherwise you will get zero for this answer.
# Make sure your code has run and the answer is correct *before* submitting your notebook for marking.
import numpy as np
import scipy as sp
def posterior(Y, W, sigma2):
    Y_cent = Y - Y.mean(0)
    # Compute posterior over X
    C_x = 
    mu_x = 
    return mu_x, C_x



## Numerically Stable and Efficient Version

Just as we saw for and computation of a matrix such as
$\mathbf{Y}^\top\mathbf{Y}$ (or its centred version) can be a bad idea
in terms of loss of numerical accuracy. Fortunately, we can find the
eigenvalues and eigenvectors of the matrix $\mathbf{Y}^\top\mathbf{Y}$
without direct computation of the matrix. This can be done with the
[*singular value
decomposition*](http://en.wikipedia.org/wiki/Singular_value_decomposition).
The singular value decompsition takes a matrix, $\mathbf{Z}$ and
represents it in the form, $$
\mathbf{Z} = \mathbf{U}\boldsymbol{\Lambda}\mathbf{V}^\top
$$ where $\mathbf{U}$ is a matrix of orthogonal vectors in the columns,
meaning $\mathbf{U}^\top\mathbf{U} = \mathbf{I}$. It has the same number
of rows and columns as $\mathbf{Z}$. The matrices $\mathbf{\Lambda}$ and
$\mathbf{V}$ are both square with dimensionality given by the number of
columns of $\mathbf{Z}$. The matrix $\mathbf{\Lambda}$ is *diagonal* and
$\mathbf{V}$ is an orthogonal matrix so
$\mathbf{V}^\top\mathbf{V} = \mathbf{V}\mathbf{V}^\top = \mathbf{I}$.
The eigenvalues of the matrix $\mathbf{Y}^\top\mathbf{Y}$ are then given
by the singular values of the matrix $\mathbf{Y}^\top$ squared and the
eigenvectors are given by $\mathbf{U}$.

## Solution for $\mathbf{W}$

Given the singular value decomposition of $\mathbf{Y}$ then we have $$
\mathbf{W}=
\mathbf{U}\mathbf{L}\mathbf{R}^\top
$$ where $\mathbf{R}$ is an arbitrary rotation matrix. This implies that
the posterior is given by $$
\mathbf{C}_x =
\left[\sigma^{-2}\mathbf{R}\mathbf{L}^2\mathbf{R}^\top + \mathbf{I}\right]^{-1}
$$ because $\mathbf{U}^\top \mathbf{U} = \mathbf{I}$. Since, by
convention, we normally take $\mathbf{R} = \mathbf{I}$ to ensure that
the principal components are orthonormal we can write $$
\mathbf{C}_x = \left[\sigma^{-2}\mathbf{L}^2 +
\mathbf{I}\right]^{-1}
$$ which implies that $\mathbf{C}_x$ is actually diagonal with elements
given by $$
c_i = \frac{\sigma^2}{\sigma^2 + \ell^2_i}
$$ and allows us to write $$
\boldsymbol{ \mu}_x = [\mathbf{L}^2 + \sigma^2
\mathbf{I}]^{-1} \mathbf{L} \mathbf{U}^\top \mathbf{ y}_{i, :}
$$ $$
\boldsymbol{ \mu}_x = \mathbf{D}\mathbf{U}^\top \mathbf{ y}_{i, :}
$$ where $\mathbf{D}$ is a diagonal matrix with diagonal elements given
by $d_{i} = \frac{\ell_i}{\sigma^2 + \ell_i^2}$.

In [ ]:
import scipy as sp
import numpy as np

In [ ]:
from mlai import ppca_svd
import inspect
file_path = inspect.getfile(ppca_svd)

In [ ]:
%load -s ppca_svd {file_path}

In [ ]:
from mlai import ppca_posterior
import inspect
file_path = inspect.getfile(ppca_posterior)

In [ ]:
%load -s ppca_posterior {file_path}

In [ ]:
ppca = ppca_svd
posterior = ppca_posterior

## Scikit-learn implementation PCA

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/oil-flow-sklearn-pca.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/oil-flow-sklearn-pca.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We’ve implemented PCA as part of supporting the learning process, but in
practice we can use the `scikit-learn` implementation. Let’s try it on
the oil flow data.

## Oil Flow Data

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/oil-flow-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

This data set is from a physics-based simulation of oil flow in a
pipeline. The data was generated as part of a project to determine the
fraction of oil, water and gas in North Sea oil pipes (Bishop and James,
1993).

In [ ]:
import pods

In [ ]:
data = pods.datasets.oil()

The data consists of 1000 12-dimensional observations of simulated oil
flow in a pipeline. Each observation is labelled according to the
multi-phase flow configuration (homogeneous, annular or laminar).

In [ ]:
# Convert data["Y"] from [1, -1, -1] in each row to rows of 0 or 1 or 2
Y = data["Y"]
# Find rows with 1 in first column (class 0)
class0 = (Y[:, 0] == 1).astype(int) * 0
# Find rows with 1 in second column (class 1) 
class1 = (Y[:, 1] == 1).astype(int) * 1
# Find rows with 1 in third column (class 2)
class2 = (Y[:, 2] == 1).astype(int) * 2
# Combine into single array of class labels 0,1,2
labels = class0 + class1 + class2

The data is returned as a dictionary containing training and test inputs
(‘X’, ‘Xtst’), training and test labels (‘Y’, ‘Ytst’), and the names of
the features.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai
import numpy as np

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
# Plot first two dimensions of the data
class_names = ["homogeneous", "annular", "stratified"]
colors = ['r', 'g', 'b']

for i, name in enumerate(class_names):
    idx = labels == i
    ax.plot(data['X'][idx, 0], data['X'][idx, 1], colors[i] + '.', 
            markersize=10, label=name)
ax.set_xlabel('1st dimension')
ax.set_ylabel('2nd dimension')
ax.legend()

mlai.write_figure('oil-flow-data.svg', directory='./datasets')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/datasets/oil-flow-data.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Visualization of the first two dimensions of the oil flow
data from Bishop and James (1993)</i>

As normal we include the citation information for the data.

In [ ]:
print(data['citation'])

And extra information about the data is included, as standard, under the
keys `info` and `details`.

In [ ]:
print(data['details'])

In [ ]:
X = data['X']
Y = data['Y']

In [ ]:
cmd = install_command('scikit-learn')

In [ ]:
%system {cmd}

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
pca.fit(X)
X_pca = pca.transform(X)

In [ ]:
from matplotlib import pyplot as plt
import mlai
from mlai import plot

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
class_names = ["homogeneous", "annular", "stratified"]
for i, name in enumerate(class_names):
    ax.scatter(X_pca[labels==i, 0], X_pca[labels==i, 1], label=name)
ax.set_xlabel('First principal component')
ax.set_ylabel('Second principal component')

mlai.write_figure("oil-flow-pca-sklearn.svg", directory='./dimred')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/oil-flow-pca_sklearn.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>PCA of the oil flow data. The flow is homogeneous, annular
and stratified.</i>

## Difficulty for Probabilistic Approaches

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/non-linear-difficulty.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/non-linear-difficulty.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import mlai.plot as plot

In [ ]:
plot.non_linear_difficulty_plot_3(diagrams='./dimred/')

The challenge for composition of probabilistic models is that you need
to propagate a probability densities through non linear mappings. This
allows you to create broader classes of probability density.
Unfortunately it renders the resulting densities *intractable*.

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/nonlinear-mapping-3d-plot.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>A two dimensional grid mapped into three dimensions to form a
two dimensional manifold.</i>

In [ ]:
plot.non_linear_difficulty_plot_2(diagrams='./dimred/')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/nonlinear-mapping-2d-plot.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>A one dimensional line mapped into two dimensions by two
separate independent functions. Each point can be mapped exactly through
the mappings.</i>

In [ ]:
cmd = install_command('daft')

In [ ]:
%system {cmd}

In [ ]:
plot.non_linear_difficulty_plot_1(diagrams='./dimred')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/gaussian-through-nonlinear.svg" class="" width="100%" style="vertical-align:middle;">

Figure: <i>A Gaussian density over the input of a non linear function
leads to a very non Gaussian output. Here the output is multimodal.</i>

## Getting Started and Downloading Data

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/gplvm-tutorial-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/gplvm-tutorial-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import numpy as np
import GPy
import string

The following code is for plotting and to prepare the bigger models for
later useage. If you are interested, you can have a look, but this is
not essential.

In [ ]:
from matplotlib import pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
colors = ["#3FCC94", "#DD4F23", "#C6D63B", "#D44271", 
          "#E4A42C", "#4F9139", "#6DDA4C", "#85831F", 
          "#B36A29", "#CF4E4A"]

In [ ]:
def plot_model(X, which_dims, labels):
    fig, ax = plt.subplots(figsize=plot.big_figsize)
    X = X[:,which_dims]
    ulabs = []
    for lab in labels:
        if not lab in ulabs:
            ulabs.append(lab)
            pass
        pass
    for i, lab in enumerate(ulabs):
        ax.scatter(*X[labels==lab].T,marker='o',color=colors[i],label=lab)
        pass
    pass

For this lab, we’ll use a data set containing all handwritten digits
from $0 \cdots 9$ handwritten, provided by de Campos et al. (2009). We
will only use some of the digits for the demonstrations in this lab
class, but you can edit the code below to select different subsets of
the digit data as you wish.

In [ ]:
import pods

In [ ]:
which = [0,1,2,6,7,9] # which digits to work on
data = pods.datasets.decampos_digits(which_digits=which)
Y = data['Y']
labels = data['str_lbls']

You can try to plot some of the digits using `plt.matshow` (the digit
images have size `16x16`).

## Principal Component Analysis

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/gplvm-tutorial.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/gplvm-tutorial.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Principal component analysis (PCA) finds a rotation of the observed
outputs, such that the rotated principal component (PC) space maximizes
the variance of the data observed, sorted from most to least important
(most to least variable in the corresponding PC).

In order to apply PCA in an easy way, we have included a PCA module in
pca.py. You can import the module by import \<path.to.pca\> (without the
ending .py!). To run PCA on the digits we have to reshape (Hint:
np.reshape ) digits .

-   What is the right shape $n\times d$ to use?

We will call the reshaped observed outputs $\mathbf{Y}$ in the
following.

In [ ]:
Yn = Y#Y-Y.mean()

Now let’s run PCA on the reshaped dataset $\mathbf{Y}$:

In [ ]:
from GPy.util import pca
p = pca.PCA(Y) # create PCA class with digits dataset

The resulting plot will show the lower dimensional representation of the
digits in 2 dimensions.

In [ ]:

p.plot_fracs(20) # plot first 20 eigenvalue fractions
p.plot_2d(Y,labels=labels.flatten(), colors=colors)
plt.legend()

## Gaussian Process Latent Variable Model

The Gaussian Process Latent Variable Model (GP-LVM) (Lawrence, 2005)
embeds PCA into a Gaussian process framework, where the latent inputs
$\mathbf{Z}$ are learnt as hyperparameters and the mapping variables
$\mathbf{W}$ are integrated out. The advantage of this interpretation is
it allows PCA to be generalized in a non linear way by replacing the
resulting *linear* covariance witha non linear covariance. But first,
let’s see how GPLVM is equivalent to PCA using an automatic relevance
determination (ARD, see e.g. Bishop (2006)) linear kernel:

In [ ]:
input_dim = 4 # How many latent dimensions to use
kernel = GPy.kern.Linear(input_dim, ARD=True) # ARD kernel
m = GPy.models.GPLVM(Yn, input_dim=input_dim, kernel=kernel)

m.optimize(messages=1, max_iters=1000) # optimize for 1000 iterations

In [ ]:
m.kern.plot_ARD()
plot_model(m.X, m.linear.variances.argsort()[-2:], labels.flatten())
plt.legend()

As you can see the solution with a linear kernel is the same as the PCA
solution with the exception of rotational changes and axis flips.

For the sake of time, the solution you see was only running for 1000
iterations, thus it might not be converged fully yet. The GP-LVM
proceeds by iterative optimization of the *inputs* to the covariance. As
we saw in the lecture earlier, for the linear covariance, these latent
points can be optimized with an eigenvalue problem, but generally, for
non-linear covariance functions, we are obliged to use gradient based
optimization.

## CMU Mocap Database

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_datasets/includes/cmu-mocap-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_datasets/includes/cmu-mocap-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Motion capture data from the CMU motion capture data base (CMU Motion
Capture Lab, 2003).

In [ ]:
import pods

You can download any subject and motion from the data set. Here we will
download motion `01` from subject `35`.

In [ ]:
subject='35' 
motion=['01']

In [ ]:
data = pods.datasets.cmu_mocap(subject, motion)

The data dictionary contains the keys ‘Y’ and ‘skel’, which represent
the data and the skeleton..

In [ ]:
data['Y'].shape

The data was used in the hierarchical GP-LVM paper (Lawrence and Moore,
2007) in an experiment that was also recreated in the Deep Gaussian
process paper (Damianou and Lawrence, 2013).

In [ ]:
print(data['citation'])

And extra information about the data is included, as standard, under the
keys `info` and `details`.

In [ ]:
print(data['info'])
print()
print(data['details'])

## CMU Motion Capture GP-LVM

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/cmu-mocap-gplvm.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/cmu-mocap-gplvm.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

The original data has the figure moving across the floor during the
motion capture sequence. We can make the figure walk ‘in place’, by
setting the x, y, z positions of the root node to zero. This makes it
easier to visualize the result.

In [ ]:
# Make figure move in place.
data['Y'][:, 0:3] = 0.0

We can also remove the mean of the data.

In [ ]:
Y = data['Y']

Now we create the GP-LVM model.

In [ ]:
import GPy

In [ ]:
model = GPy.models.GPLVM(Y, 2, normalizer=True)

Now we optimize the model.

In [ ]:
model.optimize(messages=True, max_f_eval=10000)

In [ ]:
import matplotlib.pyplot as plt
import mlai
import mlai.plot as plot

In [ ]:
%matplotlib notebook

In [ ]:
import numpy as np

In [ ]:
fig, _ = plt.subplots(figsize=plot.big_wide_figsize)
latent_axes = fig.add_subplot(131)
sense_axes = fig.add_subplot(132)
viz_axes = fig.add_subplot(133, projection='3d')

model.plot_latent(ax=latent_axes)
latent_axes.set_aspect('equal')

y = model.Y[0, :]

data_show = GPy.plotting.matplot_dep.visualize.skeleton_show(y[None, :], data['skel'], viz_axes)

lvm_visualizer = GPy.plotting.matplot_dep.visualize.lvm(model.X[0].copy(), model, data_show, latent_axes=latent_axes, sense_axes=sense_axes)
mlai.write_figure(figure=fig,
                  filename='cmu-mocap-gplvm.svg', 
                  directory = './gplvm')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/cmu-mocap-gplvm.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>Gaussian process latent variable model visualisation of CMU
motion capture data set.</i>

## Example: Latent Doodle Space

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/latent-doodle-space.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/latent-doodle-space.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

[](https://vimeo.com/3235882#t=)

Figure: <i>The latent doodle space idea of Baxter and Anjyo (2006)
manages to build a smooth mapping across very sparse data.</i>

**Generalization with much less Data than Dimensions**

-   Powerful uncertainly handling of GPs leads to surprising properties.

-   Non-linear models can be used where there are fewer data points than
    dimensions *without overfitting*.

(Baxter and Anjyo, 2006)

## Example: Continuous Character Control

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/character-control.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/character-control.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

-   Graph diffusion prior for enforcing connectivity between motions.
    $$\log p(\mathbf{X}) = w_c \sum_{i,j} \log K_{ij}^d$$ with the graph
    diffusion kernel $\mathbf{K}^d$ obtain from
    $$K_{ij}^d = \exp(\beta \mathbf{H})
    \qquad \text{with} \qquad \mathbf{H} = -\mathbf{T}^{-1/2} \mathbf{L} \mathbf{T}^{-1/2}$$
    the graph Laplacian, and $\mathbf{T}$ is a diagonal matrix with
    $T_{ii} = \sum_j w(\mathbf{ x}_i, \mathbf{ x}_j)$,
    $$L_{ij} = \begin{cases} \sum_k w(\mathbf{ x}_i,\mathbf{ x}_k) & \text{if $i=j$}
    \\
    -w(\mathbf{ x}_i,\mathbf{ x}_j) &\text{otherwise.}
    \end{cases}$$ and
    $w(\mathbf{ x}_i,\mathbf{ x}_j) = || \mathbf{ x}_i - \mathbf{ x}_j||^{-p}$
    measures similarity.

Levine et al. (2012)

## Character Control: Results

In [ ]:
from IPython.lib.display import YouTubeVideo
YouTubeVideo('hr3pdDl5IAg')

Figure: <i>Character control in the latent space described the the
GP-LVM Levine et al. (2012).</i>

## Data for Blastocyst Development in Mice: Single Cell TaqMan Arrays

Now we analyze some single cell data from Guo et al. (2010). Tey
performed qPCR TaqMan array on single cells from the developing
blastocyst in mouse. The data is taken from the early stages of
development when the Blastocyst is forming. At the 32 cell stage the
data is already separated into the trophectoderm (TE) which goes onto
form the placenta and the inner cellular mass (ICM). The ICM further
differentiates into the epiblast (EPI)—which gives rise to the endoderm,
mesoderm and ectoderm—and the primitive endoderm (PE) which develops
into the amniotic sack. Guo et al. (2010) selected 48 genes for
expression measurement. They labelled the resulting cells and their
labels are included as an aide to visualization.

They first visualized their data using principal component analysis. In
the first two principal components this fails to separate the domains.
This is perhaps because the principal components are dominated by the
variation in the 64 cell systems. This in turn may be because there are
more cells from the data set in that regime, and may be because the
natural variation is greater. We first recreate their visualization
using principal component analysis.

In this notebook we will perform PCA on the original data, showing that
the different regimes do not separate.

Next we load in the data. We’ve provided a convenience function for
loading in the data with `pods`. It is loaded in as a `pandas`
DataFrame. This allows us to summarize it with the `describe` attribute.

In [ ]:
import pods

In [ ]:
data = pods.datasets.singlecell()
Y = data['Y']
Y.describe

## Principal Component Analysis

Now we follow Guo et al. (2010) in performing PCA on the data. Rather
than calling a ‘PCA routine’, here we break the algorithm down into its
steps: compute the data covariance, compute the eigenvalues and
eigenvectors and sort according to magnitude of eigenvalue. Because we
want to visualize the data, we’ve chose to compute the eigenvectors of
the *inner product matrix* rather than the covariance matrix. This
allows us to plot the eigenvalues directly. However, this is less
efficient (in this case because the number of genes is smaller than the
number of data) than computing the eigendecomposition of the covariance
matrix.

In [ ]:
import numpy as np

In [ ]:
# obtain a centred version of data.
centredY = Y - Y.mean()
# compute inner product matrix
C = np.dot(centredY,centredY.T)
# perform eigendecomposition
V, U = np.linalg.eig(C)
# sort eigenvalues and vectors according to size
ind = V.argsort()
ev = V[ind[::-1]]
U = U[:, ind[::-1]]

To visualize the result, we now construct a simple helper function. This
will ensure that the plots have the same legends as the GP-LVM plots we
use below.

In [ ]:
import GPy
import matplotlib.pyplot as plt

In [ ]:
def plot_labels(ax, x, y, labels, symbols):
    """A small helper function for plotting with labels"""
    # make sure labels are in order of input:
    ulabels = []
    for lab in labels:
        if not lab in ulabels:
            ulabels.append(lab)
    for i, label in enumerate(ulabels):
        symbol = symbols[i % len(symbols)]
        ind = labels == label
        ax.plot(x[ind], y[ind], symbol)
    ax.legend(ulabels)

## PCA Result

Now, using the helper function we can plot the results with appropriate
labels.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
plot_labels(ax, U[:, 0], U[:, 1], data['labels'], '<>^vsd')

mlai.write_figure('singlecell-data-pca.svg', directory='./datasets')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/datasets/singlecell-data-pca.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>First two principal compoents of the Guo et al. (2010)
blastocyst development data.</i>

## GP-LVM on the Data

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/singlecell-gplvm.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/singlecell-gplvm.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

<svg viewBox="0 0 200 200" style="width:15%">

<defs> <clipPath id="clip19">

<style>
circle {
  fill: black;
}
</style>

<circle cx="100" cy="100" r="100"/> </clipPath> </defs>

<title>

Max Zwiessele

</title>

<image preserveAspectRatio="xMinYMin slice" width="100%"        xlink:href="https://mlatcl.github.io/gpss/slides/diagrams/people/max-zwiessele.jpg" clip-path="url(#clip19)"/>

</svg>
<svg viewBox="0 0 200 200" style="width:15%">

<defs> <clipPath id="clip59">

<style>
circle {
  fill: black;
}
</style>

<circle cx="100" cy="100" r="100"/> </clipPath> </defs>

<title>

Oliver Stegle

</title>

<image preserveAspectRatio="xMinYMin slice" width="100%"        xlink:href="https://mlatcl.github.io/gpss/slides/diagrams/people/oliver-stegle.jpg" clip-path="url(#clip59)"/>

</svg>

Work done as a collaboration between Max Zwiessele, Oliver Stegle and
Neil D. Lawrence.

Then, we follow Buettner and Theis (2012) in applying the GP-LVM to the
data. There is a slight pathology in the result, one which they fixed by
using priors that were dependent on the developmental stage. We then
show how the Bayesian GP-LVM doesn’t exhibit those pathologies and gives
a nice results that seems to show the lineage of the cells.

They used modified prior to ensure that small differences between cells
at the same differential stage were preserved. Here we apply a standard
GP-LVM (no modified prior) to the data.

In [ ]:
import GPy

In [ ]:
kernel = GPy.kern.RBF(2)+GPy.kern.Bias(2)
model = GPy.models.GPLVM(Y.values, 2, kernel=kernel)
model.optimize(messages=True)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
model.plot_latent(ax=ax, labels=data['labels'], marker='<>^vsd')

mlai.write_figure('singlecell-gplvm.svg', directory='./gplvm')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/singlecell-gplvm.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Visualisation of the Guo et al. (2010) blastocyst development
data with the GP-LVM.</i>

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
model.kern.plot_ARD(ax=ax)

mlai.write_figure('singlecell-gplvm-ard.svg', directory='./gplvm')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/singlecell-gplvm-ard.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The ARD parameters of the GP-LVM for the Guo et al. (2010)
blastocyst development data.</i>

In [ ]:
import pods

In [ ]:
data = pods.datasets.singlecell()
Y = data['Y']

## Blastocyst Data: Isomap

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/singlecell-isomap.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/singlecell-isomap.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Isomap first builds a neighbourhood graph, and then uses distances along
this graph to approximate the geodesic distance between points. These
distances are then visualized by performing classical multidimensional
scaling (which involves computing the eigendecomposition of the centred
distance matrix). As the neighborhood size is increased to match the
data, principal component analysis is recovered (or strictly speaking,
principal coordinate analysis). The fewer the neighbors, the more
‘non-linear’ the isomap embeddings are.

In [ ]:
import sklearn.manifold

In [ ]:
n_neighbors = 10
model = sklearn.manifold.Isomap(n_neighbors=n_neighbors, n_components=2)
X = model.fit_transform(Y)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
plot_labels(ax, X[:, 0], X[:, 1], data['labels'], '<>^vsd')


mlai.write_figure('singlecell-isomap.svg', directory='./dimred')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/singlecell-isomap.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Visualisation of the Guo et al. (2010) blastocyst development
data with Isomap.</i>

In [ ]:
import pods

In [ ]:
data = pods.datasets.singlecell()
Y = data['Y']

## Blastocyst Data: Locally Linear Embedding

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_dimred/includes/singlecell-lle.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_dimred/includes/singlecell-lle.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Next we try locally linear embedding. In locally linear embedding a
neighborhood is also computed. Each point is then reconstructed by it’s
neighbors using a linear weighting. This implies a locally linear patch
is being fitted to the data in that region. These patches are
assimilated into a large $n\times n$ matrix and a lower dimensional data
set which reflects the same relationships is then sought. Quite a large
number of neighbours needs to be selected for the data to not collapse
in on itself. When a large number of neighbours is selected the
embedding is more linear and begins to look like PCA. However, the
algorithm does *not* converge to PCA in the limit as the number of
neighbors approaches $n$.

In [ ]:
import sklearn.manifold

In [ ]:
n_neighbors = 50
model = sklearn.manifold.LocallyLinearEmbedding(n_neighbors=n_neighbors, n_components=2)
X = model.fit_transform(Y)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
plot_labels(ax, X[:, 0], X[:, 1], data['labels'], '<>^vsd')

mlai.write_figure('singlecell-lle.svg', directory='./dimred')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/dimred/singlecell-lle.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Visualisation of the Guo et al. (2010) blastocyst development
data with a locally linear embedding.</i>

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

## References

Baxter, W.V., Anjyo, K.-I., 2006. Latent doodle space, in: EUROGRAPHICS.
Vienna, Austria, pp. 477–485.
<https://doi.org/10.1111/j.1467-8659.2006.00967.x>

Bishop, C.M., 2006. Pattern recognition and machine learning. springer.

Bishop, C.M., James, G.D., 1993. Analysis of multiphase flows using
dual-energy gamma densitometry and neural networks. Nuclear Instruments
and Methods in Physics Research A327, 580–593.
<https://doi.org/10.1016/0168-9002(93)90728-Z>

Buettner, F., Theis, F.J., 2012. A novel approach for resolving
differences in single-cell gene expression patterns from zygote to
blastocyst. Bioinformatics 28, i626–i632.
<https://doi.org/10.1093/bioinformatics/bts385>

CMU Motion Capture Lab, 2003. The CMU mocap database.

Damianou, A., Lawrence, N.D., 2013. Deep Gaussian processes. pp.
207–215.

de Campos, T.E., Babu, B.R., Varma, M., 2009. Character recognition in
natural images, in: Proceedings of the Fourth International Conference
on Computer Vision Theory and Applications - Volume 2: VISAPP,
(VISIGRAPP 2009). INSTICC; SciTePress, pp. 273–280.
<https://doi.org/10.5220/0001770102730280>

Guo, G., Huss, M., Tong, G.Q., Wang, C., Sun, L.L., Clarke, N.D.,
Robsonemail, P., 2010. Resolution of cell fate decisions revealed by
single-cell gene expression analysis from zygote to blastocyst.
Developmental Cell 18, 675–685.
<https://doi.org/10.1016/j.devcel.2010.02.012>

Lawrence, N.D., 2005. Probabilistic non-linear principal component
analysis with Gaussian process latent variable models. Journal of
Machine Learning Research 6, 1783–1816.

Lawrence, N.D., Moore, A.J., 2007. Hierarchical Gaussian process latent
variable models. pp. 481–488.

Levine, S., Wang, J.M., Haraux, A., Popović, Z., Koltun, V., 2012.
Continuous character control with low-dimensional embeddings. ACM
Transactions on Graphics (SIGGRAPH 2012) 31.

Roweis, S.T., n.d. EM algorithms for PCA and SPCA. pp. 626–632.

Tipping, M.E., Bishop, C.M., 1999b. Mixtures of probabilistic principal
component analysers. Neural Computation 11, 443–482.

Tipping, M.E., Bishop, C.M., 1999a. Probabilistic principal component
analysis. Journal of the Royal Statistical Society, B 6, 611–622.
<https://doi.org/doi:10.1111/1467-9868.00196>